# PokéPrice: pipeline walkthrough

This notebook runs the pipeline and explores what it produces:

1. **Setup**: connect to the project database
2. **Run the pipeline**: whole or stage by stage
3. **Pipeline health**: run log + data-quality checks
4. **Catalogue**: English cards, scraped Japanese prints, JP → EN links
5. **Reference prices**: Cardmarket vs TCGplayer
6. **Matching**: how well messy listing titles are matched (synthetic benchmark)
7. **Sold prices**: graded eBay sales, grade ladders, graded premium
8. **Card lookup**: every estimate for one card, side by side

> **Before you start:** the database must be running. In PowerShell run
> `powershell -File scripts\dev_db.ps1` from the project folder (once after every PC restart).

> **Data note:** sold-price data comes from PokemonPriceTracker, whose terms forbid redistribution.
> Notebook outputs are stripped automatically on commit (`nbstripout`), so none of it reaches GitHub.

## 1. Setup

In [ ]:
import json, subprocess, sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from common.db import engine  # noqa: E402

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# Chart colours (validated categorical palette, slots 1-3) + recessive ink
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.figsize": (9, 4.2), "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": INK, "axes.titlelocation": "left",
})


def q(sql: str, **params) -> pd.DataFrame:
    """Run a SQL query against the project database and return a DataFrame."""
    with engine().connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)


try:
    print("Connected:", q("select current_database() db, version()").iloc[0]["db"])
except Exception as exc:
    print("Database not reachable. Run scripts\\dev_db.ps1 first.\n", exc)

## 2. Run the pipeline

Nothing runs unless you set `RUN = True`. Two things to know:

* **`sold`** spends about **90 of your 100 daily PriceTracker credits**. Run it once a day at most.
* A full run takes about 1 minute (Bulbapedia's polite 5-second crawl delay is part of it).

Pick the stages you want. `None` = full pipeline.

In [ ]:
RUN = False

# Full run:            STAGES = None
# Everything but sold: STAGES = ["fx", "tcgdex", "pokemontcg", "bulbapedia", "listings",
#                                "normalize", "match", "estimate", "validate"]
STAGES = None


def run_stage(stage=None):
    cmd = [sys.executable, "pipeline.py"] + (["--stage", stage] if stage else [])
    res = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
    lines = [l for l in res.stderr.splitlines() if " stage " in l]
    print(f"$ {' '.join(cmd[1:])}  -> exit {res.returncode}")
    for l in lines:
        print("   ", l.split(" pipeline ", 1)[-1][:220])
    return res.returncode


if RUN:
    for s in (STAGES or [None]):
        if run_stage(s) != 0:
            print("Stopped: a hard stage failed. See section 3.")
            break
else:
    print("RUN is False: showing the data already in the database.")

## 3. Pipeline health

In [ ]:
runs = q("""
    select run_id, job_name, status, started_at, completed_at,
           round(extract(epoch from completed_at - started_at)::numeric, 1) as seconds,
           rows_in, rows_out, left(error, 80) as error
    from pipeline_runs order by run_id desc limit 15""")
runs

In [ ]:
dq = q("""
    select distinct on (check_name) check_name, severity, passed, failing_rows, run_at
    from dq_results order by check_name, run_at desc""").sort_values(["passed", "severity"])
print(f"Data-quality checks passing: {dq.passed.sum()} / {len(dq)}")
dq

## 4. Catalogue: English cards + scraped Japanese prints

In [ ]:
q("""
    select language, set_id, set_name, count(*) as cards,
           count(*) filter (where rarity ilike '%special%') as special_illustration_rares
    from cards group by 1, 2, 3 order by 1""")

In [ ]:
links = q("""
    select e.method, count(*) as jp_cards
    from cards c left join card_equivalents e using (card_id)
    where c.language = 'ja' group by 1 order by 2 desc""")
print("How each Japanese print was linked to its English equivalent:")
links

In [ ]:
# Example: the same card in both languages (different numbers, different products)
q("""
    select jp.card_id as jp_card, jp.local_id || '/' || jp.set_official_count as jp_number,
           en.card_id as en_card, en.local_id || '/' || en.set_official_count as en_number,
           jp.name, jp.rarity, e.method
    from card_equivalents e
    join cards jp on jp.card_id = e.card_id
    join cards en on en.card_id = e.equivalent_card_id
    where jp.rarity = 'Special illustration rare' order by jp.collector_number""")

## 5. Reference prices: Cardmarket vs TCGplayer

Both are price *guides* for raw (ungraded) cards, not sales. Points above the line are cards where
TCGplayer (US) prices higher than Cardmarket (EU) after converting to EUR.

In [ ]:
cmp = q("""
    select c.name, c.local_id, cm.finish, cm.price_eur as cardmarket_eur, tp.price_eur as tcgplayer_eur
    from (select distinct on (card_id, finish) card_id, finish, price_eur from price_observations
          where source = 'cardmarket_via_tcgdex' and price_type = 'trend'
            and variant_id in (select variant_id from card_variants where not is_special)
          order by card_id, finish, observed_at desc) cm
    join (select distinct on (card_id, finish) card_id, finish, price_eur from price_observations
          where source = 'tcgplayer_via_pokemontcg' and price_type = 'market'
          order by card_id, finish, observed_at desc) tp using (card_id, finish)
    join cards c using (card_id)""")

if cmp.empty:
    print("No TCGplayer prices yet (that API is sometimes down). Re-run stage 'pokemontcg'.")
else:
    cmp[["cardmarket_eur", "tcgplayer_eur"]] = cmp[["cardmarket_eur", "tcgplayer_eur"]].astype(float)
    ratio = (cmp.tcgplayer_eur / cmp.cardmarket_eur).median()
    fig, ax = plt.subplots(figsize=(6.5, 6))
    ax.scatter(cmp.cardmarket_eur, cmp.tcgplayer_eur, s=22, color=BLUE, alpha=0.75,
               edgecolor="white", linewidth=0.8)
    lo, hi = cmp[["cardmarket_eur", "tcgplayer_eur"]].min().min(), cmp[["cardmarket_eur", "tcgplayer_eur"]].max().max()
    ax.plot([lo, hi], [lo, hi], color=MUTED, linewidth=1, linestyle="--")
    ax.text(hi, hi, "  equal price", color=MUTED, va="center", fontsize=9)
    ax.set(xscale="log", yscale="log", xlabel="Cardmarket trend (EUR)", ylabel="TCGplayer market (EUR)",
           title=f"Same card, two price guides (median ratio {ratio:.2f}×)")
    plt.show()
    print(f"{len(cmp)} card × finish pairs compared")

In [ ]:
print("Most valuable raw cards (reference estimate):")
q("""
    select c.name, c.local_id, c.rarity, e.finish, e.estimated_price as eur, e.confidence, e.n_observations
    from v_latest_estimates e join cards c using (card_id)
    where e.estimate_type = 'reference' order by e.estimated_price desc limit 10""")

## 6. Matching messy listing titles

> Measured on the **synthetic benchmark**: listing titles generated from the real catalogue with realistic
> noise and known answers. It's an upper bound until real marketplace titles are hand-labelled.

In [ ]:
metrics_file = ROOT / "data" / "metrics" / "latest_run.json"
if metrics_file.exists():
    ev = json.loads(metrics_file.read_text(encoding="utf-8"))["stages"].get("match", {}).get("result", {}).get("evaluation", {})
    print({k: v for k, v in ev.items() if k != "by_noise_kind"})
    noise = pd.DataFrame(ev.get("by_noise_kind", {})).T.assign(
        accuracy=lambda d: (d.auto_correct / d.n).round(3)).sort_values("accuracy")
    display(noise)
else:
    print("Run the full pipeline once to create data/metrics/latest_run.json")

In [ ]:
if metrics_file.exists() and not noise.empty:
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.barh(noise.index, noise.accuracy, color=BLUE, height=0.6)
    for y, v in enumerate(noise.accuracy):
        ax.text(v + 0.01, y, f"{v:.0%}", va="center", fontsize=9, color=INK)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set(xlim=(0, 1.1), xlabel="auto-matched correctly",
           title="Matching accuracy by kind of title noise")
    ax.grid(axis="y", visible=False)
    plt.show()

In [ ]:
print("Review queue: titles the matcher was unsure about (a human decides):")
q("""
    select l.title, m.card_id as best_guess, m.match_score, m.match_method
    from card_matches m join listings l using (source, listing_id)
    where m.status = 'review' order by m.match_score desc limit 10""")

## 7. Sold prices: graded eBay sales

One row per **card × grade**. The price is the *median* sold price, and confidence drops when sales are few
or the last sale is old.

In [ ]:
sold = q("""
    select e.card_id, c.name, c.local_id, e.grade, e.estimated_price as sold_eur,
           e.n_observations as sales, e.confidence
    from v_latest_estimates e join cards c using (card_id)
    where e.estimate_type = 'sales'""")
print(f"{sold.card_id.nunique()} cards, {len(sold)} card × grade sold-price estimates")
sold.pivot_table(index="grade", columns="confidence", values="card_id", aggfunc="count", fill_value=0)

In [ ]:
CARD = "sv03.5-199"   # change me: any card_id from the tables above
MIN_SALES = 10        # grades with fewer sales are listed in the table, not charted

ladder = q("""
    select e.grade, e.estimated_price as eur, e.n_observations as sales, e.confidence
    from v_latest_estimates e
    where e.card_id = :c and e.estimate_type = 'sales' order by e.n_observations desc""", c=CARD)
raw = q("""select estimated_price from v_latest_estimates
             where card_id = :c and grade = 'RAW' and estimate_type = 'reference'
             order by estimated_price desc limit 1""", c=CARD)
name = q("select name || ' ' || local_id as n from cards where card_id = :c", c=CARD).n.iloc[0]

if ladder.empty:
    print(f"No sold data for {CARD} yet: the daily batch rotates through the set (about 45 cards a day).")
else:
    ladder["eur"] = ladder.eur.astype(float)
    shown = ladder[ladder.sales >= MIN_SALES].sort_values("eur")
    fig, ax = plt.subplots(figsize=(9, 0.42 * len(shown) + 1.4))
    ax.barh(shown.grade, shown.eur, color=BLUE, height=0.62)
    for y, (eur, n) in enumerate(zip(shown.eur, shown.sales)):
        ax.text(eur, y, f"  €{eur:,.0f} · {n} sales", va="center", fontsize=9, color=INK)
    if not raw.empty:
        r = float(raw.estimated_price.iloc[0])
        ax.axvline(r, color=ORANGE, linewidth=2, linestyle="--")
        ax.text(r, len(shown) - 0.4, f" raw €{r:,.0f}", color=ORANGE, fontsize=9, va="bottom")
    ax.set(xlabel="median sold price (EUR)", xlim=(0, shown.eur.max() * 1.3),
           title=f"{name}: what each grade sells for (grades with ≥{MIN_SALES} sales)")
    ax.grid(axis="y", visible=False)
    plt.show()
    print(f"{len(ladder) - len(shown)} thinly traded grades not charted:")
    display(ladder[ladder.sales < MIN_SALES])

In [ ]:
prem = q("""
    select c.name, c.local_id, s.estimated_price as psa10_eur, r.estimated_price as raw_eur,
           round(s.estimated_price / r.estimated_price, 2) as premium
    from v_latest_estimates s
    join v_latest_estimates r on r.card_id = s.card_id and r.finish = s.finish
         and r.grade = 'RAW' and r.estimate_type = 'reference'
    join cards c on c.card_id = s.card_id
    where s.estimate_type = 'sales' and s.grade = 'PSA 10' and s.confidence = 'high'
    order by premium desc""")
if prem.empty:
    print("No high-confidence PSA 10 sales yet.")
else:
    prem["premium"] = prem.premium.astype(float)
    import numpy as np
    bins = np.geomspace(max(prem.premium.min() * 0.9, 0.5), prem.premium.max() * 1.1, 18)
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.hist(prem.premium, bins=bins, color=BLUE, edgecolor="white", linewidth=2)
    med = prem.premium.median()
    ax.axvline(med, color=ORANGE, linewidth=2)
    ax.annotate(f"median {med:.1f}×", (med, 1), xycoords=("data", "axes fraction"), xytext=(6, -14),
                textcoords="offset points", color=ORANGE, fontsize=9)
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}×"))
    ax.set(xlabel="PSA 10 sold price ÷ raw reference price (log scale)", ylabel="cards",
           title=f"Graded premium for PSA 10 ({len(prem)} cards, high-confidence sales)")
    ax.grid(axis="x", visible=False)
    plt.show()
    # Suspiciously high premiums: candidates for the anomaly-detection step (e.g. the source
    # pooling sales of special versions of a card under the base card). Not yet verified.
    suspicious = prem[prem.premium > 3 * med]
    print(f"{len(suspicious)} card(s) with a premium above 3× the median: investigate before trusting:")
    display(suspicious)

## 8. Card lookup: every estimate side by side

`reference` = price guide (raw only) · `listing` = active **asking** prices · `sales` = eBay **sold** prices.
They're never blended. `is_synthetic = True` means demo listings.

In [ ]:
def card_report(search: str):
    """Look up a card by name (partial) or card_id and show all its estimates."""
    cards = q("""select card_id, name, local_id, set_name, rarity, language from cards
                 where card_id = :s or name ilike :like order by language, collector_number""",
              s=search, like=f"%{search}%")
    if cards.empty:
        print("No card found.")
        return
    display(cards)
    ests = q("""
        select e.card_id, e.finish, e.grade, e.estimate_type, e.estimated_price as eur,
               e.n_observations as n, e.confidence, e.is_synthetic
        from v_latest_estimates e where e.card_id = any(:ids)
        order by e.card_id, e.grade = 'RAW' desc, e.grade, e.estimate_type""", ids=list(cards.card_id))
    display(ests.pivot_table(index=["card_id", "finish", "grade"], columns="estimate_type",
                             values="eur", aggfunc="first"))


card_report("Charizard ex")